Step 1

In [16]:
from pathlib import Path
import wfdb

# Anchor to the project root so the path works no matter where the kernel started.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MITDB = PROJECT_ROOT.parent / "mit-bih-arrhythmia-database-1.0.0"

record = wfdb.rdrecord(str(MITDB / "100"))
annotation = wfdb.rdann(str(MITDB / "100"), "atr")

duration_min = record.sig_len / record.fs / 60
print("Record name:", record.record_name)
print("Sampling frequency:", record.fs, "Hz")
print("Leads:", record.sig_name)
print("Units:", record.units)
print("Signal Shape:", record.p_signal.shape)
print(f"Duration: {duration_min:.1f} minutes")
print("annotations:", len(annotation.sample))
print("\nFirst 10 annotations:")
print(f"{'sample':>8} {'time(s)':>8}  symbol")
for sample, symbol in zip(annotation.sample[:10], annotation.symbol[:10]):
    print(f"{sample:>8} {sample / record.fs:>8.2f}  {symbol}")


Record name: 100
Sampling frequency: 360 Hz
Leads: ['MLII', 'V5']
Units: ['mV', 'mV']
Signal Shape: (650000, 2)
Duration: 30.1 minutes
annotations: 2274

First 10 annotations:
  sample  time(s)  symbol
      18     0.05  +
      77     0.21  N
     370     1.03  N
     662     1.84  N
     946     2.63  N
    1231     3.42  N
    1515     4.21  N
    1809     5.03  N
    2044     5.68  A
    2402     6.67  N


Step 2:

In [30]:
from collections import Counter

NON_BEAT = {"+", "~", "|", "!", "[", "]", "x", '"'}
counts = Counter(annotation.symbol)
for symbol, count in counts.most_common():
  kind = "non-beat" if symbol in NON_BEAT else "beat"
  print(symbol, count, kind)
  
total_beats = 0
for symbol, count in counts.items():
    if symbol not in NON_BEAT:
        total_beats += count
        
print("Total annotations:", sum(counts.values()))
print("Total beats:", total_beats)


N 2239 beat
A 33 beat
+ 1 non-beat
V 1 beat
Total annotations: 2274
Total beats: 2273


Step 3:

In [40]:
from pathlib import Path
import wfdb
from collections import Counter
import pandas as pd

# Anchor to the project root so the path works no matter where the kernel started.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MITDB = PROJECT_ROOT.parent / "mit-bih-arrhythmia-database-1.0.0"

DS1 = ["101", "106", "108", "109", "112", "114", "115", "116", "118", "119",
       "122", "124", "201", "203", "205", "207", "208", "209", "215", "220",
       "223", "230"]

DS2 = ["100", "103", "105", "111", "113", "117", "121", "123", "200", "202",
       "210", "212", "213", "214", "219", "221", "222", "228", "231", "232",
       "233", "234"]

BEAT_SYMBOLS = {"N", "L", "R", "e", "j", "A", "a", "J", "S", "V", "E", "F", "/", "f", "Q"}

NON_BEAT = {"+", "~", "|", "!", "[", "]", "x", '"', "s", "T", "*", "D", "=", "p", "^", "t", "u", "?", "(", ")", "@"}

def collect_counts(records, split_name):
       rows = []
       for record_id in records:
              annotation = wfdb.rdann(str(MITDB/record_id), "atr")
              counts = Counter(annotation.symbol)
              for symbol, count in counts.items():
                     if symbol not in BEAT_SYMBOLS and symbol not in NON_BEAT:
                            raise ValueError(f"Unknown symbol {symbol!r} in record {record_id}")
                     rows.append({
                            "split": split_name,
                            "record": record_id,
                            "symbol": symbol,
                            "count": count,
                            "is_beat": symbol in BEAT_SYMBOLS,
                     })
       return rows

df = pd.DataFrame(collect_counts(DS1, "DS1") + collect_counts(DS2, "DS2"))
display(df.head())

print("Rows: ", len(df))
print("Records: ", df["record"].nunique())
print("Total Annotations: ", df["count"].sum())
print("Beats: ", df.loc[df["is_beat"], "count"].sum())
print("Symbols: ", sorted(df["symbol"].unique()))


,split,record,symbol,count,is_beat
0,DS1,101,+,1,False
1,DS1,101,N,1860,True
2,DS1,101,~,4,False
3,DS1,101,|,4,False
4,DS1,101,Q,2,True


Rows:  260
Records:  44
Total Annotations:  103724
Beats:  100733
Symbols:  ['!', '"', '+', 'A', 'E', 'F', 'J', 'L', 'N', 'Q', 'R', 'S', 'V', '[', ']', 'a', 'e', 'j', 'x', '|', '~']


Step 4 (report section 2.2):

In [51]:
AAMI_MAP = {
    "N": "N",
    "L": "N",
    "R": "N",
    "e": "N",
    "j": "N",
    "A": "S",
    "a": "S",
    "J": "S",
    "S": "S",
    "V": "V",
    "E": "V",
    "F": "F",
    "/": "Q",
    "f": "Q",
    "Q": "Q",
}

CLASSES = ["N", "S", "V", "F"]

df["aami_class"] = df["symbol"].map(AAMI_MAP)
unmapped = sorted(df.loc[df["is_beat"] & df["aami_class"].isna(), "symbol"].unique())
if unmapped:
  raise ValueError(f"Beat symbol missing: {unmapped}")

df["aami_class"] = df["aami_class"].fillna("-")

symbol_table = df.pivot_table(
  index=["symbol", "is_beat", "aami_class"],
  columns="split",
  values="count",
  aggfunc="sum",
  fill_value=0,
).reset_index()

symbol_table["total"] = symbol_table["DS1"] + symbol_table["DS2"]
symbol_table = symbol_table.sort_values(["is_beat", "total"], ascending=[False, False])
symbol_table = symbol_table[["symbol", "is_beat", "aami_class", "DS1", "DS2", "total"]]
display(symbol_table)

beats = df[df["is_beat"]]

class_table = beats.pivot_table(
  index="aami_class", 
  columns="split",
  values="count",
  aggfunc="sum",
  fill_value=0,
).reindex(CLASSES + ["Q"], fill_value=0)

class_table["total"] = class_table["DS1"] + class_table["DS2"]
kept = class_table.loc[CLASSES]
class_table["% DS1"] = (100 * class_table["DS1"] / kept["DS1"].sum()).round(2)
class_table["% DS2"] = (100 * class_table["DS2"] / kept["DS2"].sum()).round(2)

display(class_table)

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)
symbol_table.to_csv(results_dir / "symbol_counts.csv", index=False)
class_table.to_csv(results_dir / "class_counts.csv")
print(f"\nSaved to {results_dir}")

split,symbol,is_beat,aami_class,DS1,DS2,total
8,N,True,N,38102,36444,74546
7,L,True,N,3949,4126,8075
10,R,True,N,3783,3476,7259
12,V,True,V,3683,3220,6903
3,A,True,S,810,1736,2546
5,F,True,F,415,388,803
17,j,True,N,16,213,229
15,a,True,S,100,50,150
4,E,True,V,105,1,106
6,J,True,S,32,51,83


split,DS1,DS2,total,% DS1,% DS2
aami_class,,,,,
N,45866,44259,90125,89.91,89.04
S,944,1837,2781,1.85,3.70
V,3788,3221,7009,7.43,6.48
F,415,388,803,0.81,0.78
Q,8,7,15,0.02,0.01



Saved to /Users/p4ffos/Projects/Study/LTU Projects (MSc)/Data_Mining/Group19_ESG_Project/results


Step 5 (report section 2.1):

In [54]:
def collect_metadata(records, split_name):
    rows = []
    for record_id in records:
        header = wfdb.rdheader(str(MITDB / record_id))
        leads = [name.strip() for name in header.sig_name]
        rows.append({
          "record": record_id, 
          "split": split_name, 
          "fs_hz": header.fs, 
          "n_channels": header.n_sig, 
          "n_samples": header.sig_len,
          "duration_min": round(header.sig_len / header.fs / 60, 2),
          "channel_0": leads[0], 
          "channel_1": leads[1] if len(leads) > 1 else "",
          "units": "/".join(header.units),
        })
    return rows
  
metadata = pd.DataFrame(collect_metadata(DS1, "DS1") + collect_metadata(DS2, "DS2"))
display(metadata)

print("records:          ", len(metadata))
print("sampling rates:   ", metadata["fs_hz"].unique())
print("channel counts:   ", metadata["n_channels"].unique())
print("units:            ", metadata["units"].unique())
print(f"duration:          {metadata['duration_min'].min():.1f} to "
      f"{metadata['duration_min'].max():.1f} min, "
      f"total {metadata['duration_min'].sum():.0f} min")

print("\nlead combinations:")
display(metadata.groupby(["channel_0", "channel_1"])["record"]
        .agg(count="count", records=lambda s: ", ".join(s))
        .sort_values("count", ascending=False))

LEAD = "MLII"
in_ch0 = metadata["channel_0"] == LEAD
in_ch1 = metadata["channel_1"] == LEAD
print(f"\n{LEAD} in channel 0: {in_ch0.sum()} records")
print(f"{LEAD} in channel 1: {in_ch1.sum()} records")
print(f"{LEAD} absent:       {metadata.loc[~(in_ch0 | in_ch1), 'record'].tolist()}")

metadata.to_csv(PROJECT_ROOT / "results" / "record_metadata.csv", index=False)

,record,split,fs_hz,n_channels,n_samples,duration_min,channel_0,channel_1,units
0,101,DS1,360,2,650000,30.09,MLII,V1,mV/mV
1,106,DS1,360,2,650000,30.09,MLII,V1,mV/mV
2,108,DS1,360,2,650000,30.09,MLII,V1,mV/mV
3,109,DS1,360,2,650000,30.09,MLII,V1,mV/mV
4,112,DS1,360,2,650000,30.09,MLII,V1,mV/mV
5,114,DS1,360,2,650000,30.09,V5,MLII,mV/mV
6,115,DS1,360,2,650000,30.09,MLII,V1,mV/mV
7,116,DS1,360,2,650000,30.09,MLII,V1,mV/mV
8,118,DS1,360,2,650000,30.09,MLII,V1,mV/mV
9,119,DS1,360,2,650000,30.09,MLII,V1,mV/mV


records:           44
sampling rates:    [360]
channel counts:    [2]
units:             <StringArray>
['mV/mV']
Length: 1, dtype: str
duration:          30.1 to 30.1 min, total 1324 min

lead combinations:


count                                            records
channel_0 channel_1                                                          
MLII      V1            38  101, 106, 108, 109, 112, 115, 116, 118, 119, 1...
          V2             2                                           103, 117
          V5             2                                           100, 123
          V4             1                                                124
V5        MLII           1                                                114


MLII in channel 0: 43 records
MLII in channel 1: 1 records
MLII absent:       []
